In [ ]:
import requests
import time
import os
import json

API_KEY = os.getenv("SCOPUS_API_KEY", "")
HEADERS = {
    'X-ELS-APIKey': API_KEY,
    'Accept': 'application/json'
}

BASE_SEARCH_URI = 'https://api.elsevier.com/content/search/scopus'
BASE_ABSTRACT_URI = 'https://api.elsevier.com/content/abstract/eid/'
AFFILIATION_NAME = 'Chulalongkorn'

def api_search(year=2010, offset=0, count=200):
    try:
        res = requests.get(
            url=BASE_SEARCH_URI,
            headers=HEADERS,
            params={
                'start': offset,
                'count': count,
                'query': f'AFFIL({AFFILIATION_NAME}) AND PUBYEAR = {year}',
            },
            timeout=10
        )

        if not res.ok:
            return None

        return res.json().get('search-results', {}).get('entry', [])

    except Exception:
        return None

def api_abstracts_retrieve(eid):
    try:
        res = requests.get(
            url=f"{BASE_ABSTRACT_URI}{eid}",
            headers=HEADERS,
            params={'view': 'FULL'},
            timeout=10
        )

        if not res.ok:
            return None

        return res.json()

    except Exception:
        return None

def write_json(path, data):
    directory = os.path.dirname(path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    with open(path, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)

if __name__ == "__main__":
    first_year = 2010
    last_year = 2017
    per_year = 3000
    per_chunk = 100

    start_time = time.time()
    total_papers = 0

    years = range(first_year, last_year + 1)

    for year in years:
        root_path = f'/Users/phuditpreechanarit/Desktop/Raw/{year}'

        for offset in range(0, per_year, per_chunk):
            elapsed = time.time() - start_time

            search_result = api_search(year=year, offset=offset, count=per_chunk)

            if not search_result:
                print(f"[{elapsed:.2f}s] Limit reached or no results for Year: {year}")
                break

            for paper in search_result:
                eid = paper.get('eid')
                if not eid:
                    continue

                abs_response = api_abstracts_retrieve(eid)

                if abs_response:
                    file_path = f"{root_path}/{eid}.json"
                    write_json(path=file_path, data=abs_response)

            count = len(search_result)
            total_papers += count
            print(f"[{elapsed:.2f}s] Year: {year} | Offset: {offset} | Saved: {count} | Total: {total_papers}")

[0.00s] Limit reached or no results for Year: 2010
[0.63s] Limit reached or no results for Year: 2011
[1.10s] Limit reached or no results for Year: 2012
[1.71s] Limit reached or no results for Year: 2013
[2.22s] Limit reached or no results for Year: 2014
[2.69s] Limit reached or no results for Year: 2015
[3.15s] Limit reached or no results for Year: 2016
[3.63s] Limit reached or no results for Year: 2017
